# Course 1: Geometric Deep Learning and Graphs
## Session 8: Curved spaces

**Goal:** embed a disease hierarchy in the Poincaré disk and compare it with a flat embedding of the same size.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. A small disease hierarchy
A hand-made tree shaped like ICD-10 (chapters, blocks, codes). The labels are illustrative, not the official list. For the real lab, load the ICD-10 file from WHO or CMS.

In [ ]:
import numpy as np, torch
tree = {
 "diseases": ["infectious", "circulatory", "respiratory", "endocrine"],
 "infectious": ["intestinal infections", "mosquito-borne viral", "tuberculosis"],
 "circulatory": ["hypertensive", "ischaemic heart", "cerebrovascular"],
 "respiratory": ["upper respiratory", "influenza and pneumonia", "chronic lower respiratory"],
 "endocrine": ["diabetes", "thyroid", "malnutrition"],
 "intestinal infections": ["cholera", "typhoid", "shigellosis"],
 "mosquito-borne viral": ["dengue", "chikungunya", "japanese encephalitis"],
 "tuberculosis": ["lung tb", "tb of other organs"],
 "hypertensive": ["essential hypertension", "hypertensive heart disease"],
 "ischaemic heart": ["angina", "heart attack"],
 "cerebrovascular": ["brain haemorrhage", "brain infarction"],
 "upper respiratory": ["common cold", "sinusitis", "tonsillitis"],
 "influenza and pneumonia": ["influenza", "viral pneumonia", "bacterial pneumonia"],
 "chronic lower respiratory": ["bronchitis", "copd", "asthma"],
 "diabetes": ["type 1 diabetes", "type 2 diabetes", "gestational diabetes"],
 "thyroid": ["hypothyroidism", "goitre"],
 "malnutrition": ["kwashiorkor", "marasmus"],
}
names = sorted(set(tree) | {c for cs in tree.values() for c in cs}); ix = {n: i for i, n in enumerate(names)}
parent = {ix[c]: ix[p] for p, cs in tree.items() for c in cs}

def ancestors(i):
    out = []
    while i in parent: i = parent[i]; out.append(i)
    return out

pairs = [(i, a) for i in range(len(names)) for a in ancestors(i)]      # (node, each of its ancestors)
print(len(names), "nodes,", len(pairs), "ancestor pairs")

### 2. Two distances
In the Poincaré disk, distance grows very fast near the edge, so there is room for the many leaves of a tree.

`d(u, v) = arcosh(1 + 2 |u - v|² / ((1 - |u|²)(1 - |v|²)))`

In [ ]:
def poincare_dist(u, v, eps=1e-6):
    uu = (u * u).sum(-1).clamp(max=1 - eps); vv = (v * v).sum(-1).clamp(max=1 - eps)
    return torch.acosh(1 + 2 * ((u - v) ** 2).sum(-1) / ((1 - uu) * (1 - vv)) + eps)

euclid_dist = lambda u, v: (u - v).norm(dim=-1)

o = torch.zeros(2)
for r in [0.5, 0.9, 0.99]:
    print(f"point at radius {r}: flat distance from centre {r:.2f}, hyperbolic distance {poincare_dist(o, torch.tensor([r, 0.])).item():.2f}")

### 3. Train both embeddings with the same loss
For each (node, ancestor) pair, the ancestor should be closer than 10 random other nodes. The hyperbolic model uses the Riemannian gradient (the ordinary gradient scaled by `(1 - |x|²)² / 4`) and is pulled back inside the disk after every step.

In [ ]:
N = len(names)
related = torch.zeros(N, N, dtype=torch.bool)
for i, a in pairs: related[i, a] = related[a, i] = True
P = torch.tensor(pairs)

def train(hyperbolic, dim=2, epochs=1500, lr=1.0, n_neg=10, seed=0):
    torch.manual_seed(seed)
    E = (torch.rand(N, dim) * 2e-3 - 1e-3).requires_grad_()
    dist = poincare_dist if hyperbolic else euclid_dist
    for epoch in range(epochs):
        u, v = P[:, 0], P[:, 1]
        neg = torch.randint(N, (len(P), n_neg))
        d_pos = dist(E[u], E[v])[:, None]
        d_neg = dist(E[u][:, None], E[neg]).masked_fill(related[u[:, None], neg] | (neg == u[:, None]), 1e4)
        loss = -torch.log_softmax(-torch.cat([d_pos, d_neg], 1), 1)[:, 0].mean()
        g, = torch.autograd.grad(loss, E)
        with torch.no_grad():
            if hyperbolic:
                g = g * ((1 - (E * E).sum(-1, keepdim=True)) ** 2 / 4)
                E -= (lr if epoch >= 20 else lr / 10) * g                 # short burn-in, as in the paper
                norm = E.norm(dim=-1, keepdim=True)
                E.copy_(torch.where(norm >= 1 - 1e-3, E / norm * (1 - 1e-3), E))
            else:
                E -= lr * g
    return E.detach()

def evaluate(E, dist):
    """For each node, rank all others by distance. Report mean rank of true relatives and mean average precision."""
    ranks, aps = [], []
    for i in range(N):
        rel = related[i]
        d = dist(E[i][None], E); d[i] = float("inf")
        order = d.argsort(); hits = rel[order].float()
        pos = torch.nonzero(hits).squeeze(-1) + 1
        ranks += [(p - k).item() for k, p in enumerate(pos, 0)]
        aps.append((torch.arange(1, len(pos) + 1) / pos).mean().item())
    return np.mean(ranks), np.mean(aps)

E_h, E_e = train(True), train(False)
(r_h, map_h), (r_e, map_e) = evaluate(E_h, poincare_dist), evaluate(E_e, euclid_dist)
print(f"Euclidean, 2-D  : mean rank {r_e:.2f}  MAP {map_e:.3f}")
print(f"Poincaré,  2-D  : mean rank {r_h:.2f}  MAP {map_h:.3f}")
assert map_h > map_e

### 4. Look at it
General terms should sit near the centre and specific codes near the edge.

In [ ]:
import matplotlib.pyplot as plt
depth = np.array([len(ancestors(i)) for i in range(N)]); radius = E_h.norm(dim=-1).numpy()
for dpt in range(4): print(f"depth {dpt}: mean radius {radius[depth == dpt].mean():.3f}")
assert radius[depth == 0].mean() < radius[depth == 3].mean()
fig, ax = plt.subplots(figsize=(7, 7)); ax.add_patch(plt.Circle((0, 0), 1, fill=False))
ax.scatter(E_h[:, 0], E_h[:, 1], c=depth)
for i in range(N):
    if depth[i] <= 1: ax.annotate(names[i], E_h[i].tolist(), fontsize=8)
ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05); fig.savefig("poincare_disk.png")

### 5. Covariance matrices live on a curved space too
For EEG, each trial gives a covariance matrix (symmetric positive definite). The log-Euclidean distance respects that geometry.

In [ ]:
def spd_log(S):
    w, V = np.linalg.eigh(S); return (V * np.log(w)) @ V.T
A_, B_ = np.diag([1.0, 100.0]), np.diag([100.0, 1.0])
print("flat mean has determinant", np.linalg.det((A_ + B_) / 2), "although both inputs have determinant 100 (the swelling effect)")
w, V = np.linalg.eigh((spd_log(A_) + spd_log(B_)) / 2)
print("log-Euclidean mean has determinant", round(np.linalg.det((V * np.exp(w)) @ V.T), 6))

### TODO
1. Train both models in 5 and 10 dimensions. At what size does the flat embedding catch up?
2. Load a real ICD-10 chapter and repeat. Report MAP for both.
3. Explain in three sentences why a tree needs exponentially growing room and where the disk provides it.